In [3]:
# Instalar a biblioteca xlrd necessária para ler arquivos .xls
import sys
try:
    import xlrd
    print(" xlrd já está instalado")
except ImportError:
    print("Instalando xlrd...")
    !pip install xlrd
    import xlrd
    print(" xlrd instalado com sucesso!")


 xlrd já está instalado


In [1]:
import pandas as pd
import os

# Lista com os nomes dos 7 arquivos CSV na pasta dados
arquivos_csv = [
    '../Dados/PNAD_COVID_052020.csv',
    '../Dados/PNAD_COVID_062020.csv',
    '../Dados/PNAD_COVID_072020.csv',
    '../Dados/PNAD_COVID_082020.csv',
    '../Dados/PNAD_COVID_092020.csv',
    '../Dados/PNAD_COVID_102020.csv',
    '../Dados/PNAD_COVID_112020.csv'
]

# Dicionário para armazenar os DataFrames carregados
dfs = {}

# Carregar cada arquivo CSV
for arquivo in arquivos_csv:
    if os.path.exists(arquivo):
        try:
            # Carregar o arquivo CSV
            df = pd.read_csv(arquivo, low_memory=False)
            dfs[arquivo] = df
            print(f"Carregado com sucesso!")
            print(f"  Dimensoes: {df.shape[0]} linhas x {df.shape[1]} colunas")
        except Exception as e:
            print(f"Erro ao carregar: {e}")
    else:
        print(f"Arquivo nao encontrado: {arquivo}")

print(f"\n{'='*60}")
print(f"Total de arquivos carregados: {len(dfs)}")
print(f"{'='*60}")


Carregado com sucesso!
  Dimensoes: 349306 linhas x 114 colunas
Carregado com sucesso!
  Dimensoes: 381270 linhas x 114 colunas
Carregado com sucesso!
  Dimensoes: 384166 linhas x 145 colunas
Carregado com sucesso!
  Dimensoes: 386520 linhas x 145 colunas
Carregado com sucesso!
  Dimensoes: 387298 linhas x 145 colunas
Carregado com sucesso!
  Dimensoes: 380461 linhas x 145 colunas
Carregado com sucesso!
  Dimensoes: 381438 linhas x 148 colunas

Total de arquivos carregados: 7


In [2]:
# Mostrar as primeiras linhas de cada arquivo carregado
for nome_arquivo, df in dfs.items():
    print(f"Formato de: {nome_arquivo}")
    print(f"Shape: {df.shape[0]} linhas x {df.shape[1]} colunas\n")


Formato de: ../Dados/PNAD_COVID_052020.csv
Shape: 349306 linhas x 114 colunas

Formato de: ../Dados/PNAD_COVID_062020.csv
Shape: 381270 linhas x 114 colunas

Formato de: ../Dados/PNAD_COVID_072020.csv
Shape: 384166 linhas x 145 colunas

Formato de: ../Dados/PNAD_COVID_082020.csv
Shape: 386520 linhas x 145 colunas

Formato de: ../Dados/PNAD_COVID_092020.csv
Shape: 387298 linhas x 145 colunas

Formato de: ../Dados/PNAD_COVID_102020.csv
Shape: 380461 linhas x 145 colunas

Formato de: ../Dados/PNAD_COVID_112020.csv
Shape: 381438 linhas x 148 colunas



In [18]:
# Transformar o arquivo PNAD_COVID_072020.csv
# 1. Transformar coluna Ano em colunas dummy (uma coluna para cada ano)
# 2. Transformar apenas as colunas específicas listadas em codigos.ipynb

# Carregar o arquivo específico
df_072020 = dfs['../Dados/PNAD_COVID_102020.csv'].copy()

print("Shape original:", df_072020.shape)
print("\nColunas originais:", list(df_072020.columns)[:10], "...")

# Lista de colunas específicas a serem transformadas (do arquivo codigos.ipynb)
colunas_especificas = [
    'A004',
    'A005',
    'B0011', 'B0012', 'B0013', 'B0015', 'B0018', 'B00111', 'B00112', 
        'B0014', 'B0016', 'B0017', 'B0019', 'B00110', 'B00113',
    'B002', 
    'B0031', 'B0032', 'B0033', 'B0034', 'B0035', 'B0036', 'B0037',
    'B0041', 'B0042', 'B0043', 'B0044', 'B0045', 'B0046',
    'B009B', 
    'B009D', 
    'B009F',
    'B0101', 'B0102', 'B0103', 'B0104', 'B0105', 'B0106',
    'B011',
    'C001', 
    'C002', 
    'C01012', 'C01022', 'C011A12', 'C011A22', 
    'C012', 
    'C013',
    'D0013', 'D0023', 'D0033', 'D0043', 'D0053', 'D0063', 'D0073',
    'E001',
    'F001',
    'F002A1', 'F002A2', 'F002A3', 'F002A4', 'F002A5'
]

# Filtrar apenas as colunas que existem no DataFrame
colunas_vdf = [col for col in colunas_especificas if col in df_072020.columns]
colunas_nao_encontradas = [col for col in colunas_especificas if col not in df_072020.columns]

# Identificar todas as colunas que começam com V, D, F, A, C, B, E (para removê-las)
import re
# Padrão para colunas que começam com V, D, F, A, C, B, E seguido de números/letras
# Exclui palavras completas como 'Ano' e 'CAPITAL'
padrao_vdf = re.compile(r'^(V|D|F|A|C|B|E)[\dA-Z]')

# Todas as colunas V, D, F, A, C, B, E no DataFrame (excluindo 'Ano' e 'CAPITAL')
todas_colunas_vdf = [col for col in df_072020.columns 
                     if padrao_vdf.match(col) and col not in ['Ano', 'CAPITAL']]

# Colunas V, D, F, A, C, B, E que NÃO estão na lista específica (devem ser removidas)
colunas_para_remover = [col for col in todas_colunas_vdf if col not in colunas_vdf]

# Colunas de identificação (não começam com V, D, F, A, C, B, E seguido de números/letras)
# Inclui 'Ano' e 'CAPITAL' explicitamente
colunas_outras = [col for col in df_072020.columns 
                  if (not padrao_vdf.match(col) or col in ['Ano', 'CAPITAL']) 
                  and col not in colunas_vdf]

print(f"\nColunas específicas a transformar: {len(colunas_vdf)}")
print(f"Exemplos: {colunas_vdf[:5]}")
if colunas_nao_encontradas:
    print(f"\nAtenção: {len(colunas_nao_encontradas)} colunas não encontradas no DataFrame:")
    print(f"Exemplos: {colunas_nao_encontradas[:5]}")
print(f"\nColunas V/D/F/A/C/B/E a serem removidas (não estão na lista): {len(colunas_para_remover)}")
print(f"Exemplos: {colunas_para_remover[:5]}")
print(f"\nColunas de identificação (mantidas): {len(colunas_outras)}")
print(f"Exemplos: {colunas_outras[:5]}")

# Remover as colunas V, D, F, A, C, B, E que não estão na lista específica
df_072020 = df_072020.drop(columns=colunas_para_remover)
print(f"\nApós remover colunas não desejadas: {df_072020.shape}")

# 1. Transformar coluna Ano em colunas dummy
anos_unicos = df_072020['Ano'].unique()
print(f"\nAnos únicos encontrados: {anos_unicos}")
for ano in anos_unicos:
    df_072020[f'Ano_{int(ano)}'] = (df_072020['Ano'] == ano).astype(int)
df_072020 = df_072020.drop(columns=['Ano'])

print(f"\nApós criar colunas dummy de Ano: {df_072020.shape}")

# 2. Transformar apenas as colunas específicas em linhas usando melt
# Manter as outras colunas como identificadores
# Filtrar apenas colunas que existem no DataFrame (remover 'Ano' que já foi removido)
colunas_id_vars = [col for col in colunas_outras if col in df_072020.columns]
id_vars = colunas_id_vars + [col for col in df_072020.columns if col.startswith('Ano_')]

# Garantir que todas as colunas em colunas_vdf existem no DataFrame
colunas_vdf_final = [col for col in colunas_vdf if col in df_072020.columns]

# print(f"\nFazendo melt das {len(colunas_vdf_final)} colunas específicas...")
df_transformado = df_072020
# df_transformado = pd.melt(
#     df_072020,
#     id_vars=id_vars,
#     value_vars=colunas_vdf_final,
#     var_name='Variavel',
#     value_name='Valor'
# )

print(f"\nShape após transformação: {df_transformado.shape}")
print(f"\nPrimeiras linhas do resultado:")
display(df_transformado.head(3))

# colunas_para_ver = ['C011A12', 'C011A22', 'C01012', 'C01022']
# display(df_transformado[colunas_para_ver].head(20))


Shape original: (380461, 145)

Colunas originais: ['Ano', 'UF', 'CAPITAL', 'RM_RIDE', 'V1008', 'V1012', 'V1013', 'V1016', 'Estrato', 'UPA'] ...

Colunas específicas a transformar: 61
Exemplos: ['A004', 'A005', 'B0011', 'B0012', 'B0013']

Colunas V/D/F/A/C/B/E a serem removidas (não estão na lista): 77
Exemplos: ['V1008', 'V1012', 'V1013', 'V1016', 'V1022']

Colunas de identificação (mantidas): 7
Exemplos: ['Ano', 'UF', 'CAPITAL', 'RM_RIDE', 'Estrato']

Após remover colunas não desejadas: (380461, 68)

Anos únicos encontrados: [2020]

Após criar colunas dummy de Ano: (380461, 68)

Shape após transformação: (380461, 68)

Primeiras linhas do resultado:


,UF,CAPITAL,RM_RIDE,Estrato,UPA,posest,A004,A005,B0011,B0012,...,D0063,D0073,E001,F001,F002A1,F002A2,F002A3,F002A4,F002A5,Ano_2020
0,11,11.0,NaN,1110011,110015970,1114,4,5,2,2,...,NaN,NaN,3,1,1,1,1,2,1,1
1,11,11.0,NaN,1110011,110015970,1124,4,7,2,2,...,NaN,NaN,3,1,1,1,1,2,1,1
2,11,11.0,NaN,1110011,110015970,1112,4,2,2,2,...,NaN,NaN,3,1,1,1,1,2,1,1


In [15]:
import numpy as np

# Definição das Listas de Colunas
cols_b1_principais = ['B0011', 'B0012', 'B0013', 'B0015', 'B0018', 'B00111']
cols_b1_outros = ['B0014', 'B0016', 'B0017', 'B0019', 'B00110', 'B00112', 'B00113']
cols_b3_isolamento = ['B0031']
cols_b3_orientacao = ['B0032', 'B0034', 'B0035', 'B0036']
cols_b3_automedicacao = ['B0033']
cols_b3_outra = ['B0037']
cols_b4 = ['B0041', 'B0042', 'B0043', 'B0044', 'B0045', 'B0046']
cols_b9 = ['B009B', 'B009D', 'B009F']
cols_b10 = ['B0101', 'B0102', 'B0103', 'B0104', 'B0105', 'B0106']
cols_c10 = ['C01012', 'C01022']
cols_c11 = ['C011A12', 'C011A22']
cols_d1 = ['D0013', 'D0023', 'D0033', 'D0043', 'D0053', 'D0063', 'D0073']
cols_f2a = ['F002A1', 'F002A2', 'F002A3', 'F002A4', 'F002A5']

# Verificar quais colunas existem no DF
b1_princ_validas = [c for c in cols_b1_principais if c in df_transformado.columns]
b1_outros_validas = [c for c in cols_b1_outros if c in df_transformado.columns]
b3_isol_validas = [c for c in cols_b3_isolamento if c in df_transformado.columns]
b3_orien_validas = [c for c in cols_b3_orientacao if c in df_transformado.columns]
b3_auto_validas = [c for c in cols_b3_automedicacao if c in df_transformado.columns]
b3_outra_validas = [c for c in cols_b3_outra if c in df_transformado.columns]
b4_validas = [c for c in cols_b4 if c in df_transformado.columns]
b9_validas = [c for c in cols_b9 if c in df_transformado.columns]
b10_validas = [c for c in cols_b10 if c in df_transformado.columns]
c10_validas = [c for c in cols_c10 if c in df_transformado.columns]
c11_validas = [c for c in cols_c11 if c in df_transformado.columns]
d1_validas = [c for c in cols_d1 if c in df_transformado.columns]
f2a_validas = [c for c in cols_f2a if c in df_transformado.columns]

# Função de Cálculo
def calcular_grupo(df, colunas_alvo):
    if not colunas_alvo:
        return 2
    tem_sintoma = df[colunas_alvo].isin([1, 1.0, '1']).any(axis=1)
    return np.where(tem_sintoma, 1, 2)

def calcular_soma_valores(df, colunas_alvo):
    dados = df[colunas_alvo].copy()
    dados = dados.fillna(0.0)
    return dados.sum(axis=1)

# Criação das Novas Colunas
print("Calculando novas colunas...")
df_transformado['B1_Sintomas_Principais'] = calcular_grupo(df_transformado, b1_princ_validas)
df_transformado['B1_Outros_Sintomas'] = calcular_grupo(df_transformado, b1_outros_validas)

df_transformado['B3_Isolamento'] = calcular_grupo(df_transformado, b3_isol_validas)
df_transformado['B3_Buscou_Orientacao'] = calcular_grupo(df_transformado, b3_orien_validas)
df_transformado['B3_Automedicacao'] = calcular_grupo(df_transformado, b3_auto_validas)
df_transformado['B3_Outra'] = calcular_grupo(df_transformado, b3_outra_validas)

df_transformado['B4'] = calcular_grupo(df_transformado, b4_validas)

df_transformado['B9'] = calcular_grupo(df_transformado, b9_validas)
df_transformado['B10'] = calcular_grupo(df_transformado, b10_validas)

df_transformado['C10'] = calcular_soma_valores(df_transformado, c10_validas)
df_transformado['C11'] = calcular_soma_valores(df_transformado, c11_validas)
df_transformado['D1'] = calcular_soma_valores(df_transformado, d1_validas)

df_transformado['F2A'] = calcular_grupo(df_transformado, f2a_validas)

# Limpeza
colunas_originais = (cols_b1_principais + cols_b1_outros + 
                     cols_b3_isolamento + cols_b3_orientacao + cols_b3_automedicacao + cols_b3_outra + 
                     cols_b4 + 
                     cols_b9 +
                     cols_b10 + 
                     cols_c10 + cols_c11 + 
                     cols_d1 +
                     cols_f2a
                     )

# Filtra apenas o que existe para não dar erro se rodar duas vezes
colunas_para_dropar = [c for c in colunas_originais if c in df_transformado.columns]
df_transformado = df_transformado.drop(columns=colunas_para_dropar)

print("Estrutura do DataFrame Final (Info):")
df_transformado.info()
print(df_transformado['B9'].value_counts(dropna=False))

colunas_para_ver = ['C10', 'C11', 'D1']
display(df_transformado[colunas_para_ver].head(20))

Calculando novas colunas...
Estrutura do DataFrame Final (Info):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380461 entries, 0 to 380460
Data columns (total 30 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   UF                      380461 non-null  int64  
 1   CAPITAL                 89602 non-null   float64
 2   RM_RIDE                 120300 non-null  float64
 3   Estrato                 380461 non-null  int64  
 4   UPA                     380461 non-null  int64  
 5   posest                  380461 non-null  int64  
 6   A004                    380461 non-null  int64  
 7   A005                    380461 non-null  int64  
 8   B002                    13958 non-null   float64
 9   B011                    380461 non-null  int64  
 10  C001                    314115 non-null  float64
 11  C002                    173010 non-null  float64
 12  C012                    136044 non-null  float64
 13  C013     

,C10,C11,D1
0,1045.0,1045.0,1200.0
1,1045.0,1045.0,1200.0
2,0.0,0.0,1200.0
3,0.0,0.0,1200.0
4,0.0,0.0,1045.0
5,200.0,85.0,1045.0
6,1500.0,600.0,600.0
7,0.0,0.0,600.0
8,0.0,0.0,600.0
9,0.0,0.0,600.0


In [19]:
# Salvar o DataFrame transformado em CSV
nome_arquivo_saida = 'PNAD_COVID_102020_transformado.csv'

print(f"Salvando arquivo: {nome_arquivo_saida}")
print(f"Shape do DataFrame: {df_transformado.shape}")

try:
    df_transformado.to_csv(nome_arquivo_saida, index=False, encoding='utf-8')
    print(f"\nArquivo salvo com sucesso: {nome_arquivo_saida}")
    print(f"Tamanho aproximado: {os.path.getsize(nome_arquivo_saida) / (1024*1024):.2f} MB")
except Exception as e:
    print(f"\nErro ao salvar arquivo: {e}")


Salvando arquivo: PNAD_COVID_102020_transformado.csv
Shape do DataFrame: (380461, 68)

Arquivo salvo com sucesso: PNAD_COVID_102020_transformado.csv
Tamanho aproximado: 50.70 MB
